In [32]:
import pandas as pd 
path = r"C:\Users\91766\Downloads\airline_booking_dataset\7_segments.csv"
df = pd.read_csv(path, dtype = {"ticket_no": str, "seat_no": str})
df

,ticket_no,flight_id,fare_conditions,price,seat_no,boarding_no,boarding_time
0,1000000000000,160,Economy,8522.02,45E,1.0,2025-03-29 18:51:00
1,1000000000001,735,Business,36970.69,NaN,NaN,NaN
2,1000000000002,456,Economy,5145.47,NaN,NaN,NaN
3,1000000000002,13,Economy,3414.06,18C,1.0,2025-01-25 02:01:00
4,1000000000003,116,Economy,3399.07,NaN,NaN,NaN
...,...,...,...,...,...,...,...
6253,1000000004801,650,Economy,3732.22,NaN,NaN,NaN
6254,1000000004802,118,Economy,8126.78,NaN,NaN,NaN
6255,1000000004803,129,Economy,6433.85,32F,6.0,2025-10-02 03:00:00
6256,1000000004804,424,Economy,8561.65,NaN,NaN,NaN


In [33]:
print(df.shape)

(6258, 7)


In [34]:
df.head()

,ticket_no,flight_id,fare_conditions,price,seat_no,boarding_no,boarding_time
0,1000000000000,160,Economy,8522.02,45E,1.0,2025-03-29 18:51:00
1,1000000000001,735,Business,36970.69,NaN,NaN,NaN
2,1000000000002,456,Economy,5145.47,NaN,NaN,NaN
3,1000000000002,13,Economy,3414.06,18C,1.0,2025-01-25 02:01:00
4,1000000000003,116,Economy,3399.07,NaN,NaN,NaN


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6258 entries, 0 to 6257
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ticket_no        6258 non-null   object 
 1   flight_id        6258 non-null   int64  
 2   fare_conditions  6258 non-null   object 
 3   price            6258 non-null   float64
 4   seat_no          3083 non-null   object 
 5   boarding_no      3083 non-null   float64
 6   boarding_time    3083 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 342.4+ KB


In [36]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns

Index(['ticket_no', 'flight_id', 'fare_conditions', 'price', 'seat_no',
       'boarding_no', 'boarding_time'],
      dtype='object')

In [37]:
print(df.isnull().sum())

ticket_no             0
flight_id             0
fare_conditions       0
price                 0
seat_no            3175
boarding_no        3175
boarding_time      3175
dtype: int64


In [38]:
print(df[["seat_no", "boarding_no", "boarding_time"]].isnull().value_counts())

seat_no  boarding_no  boarding_time
True     True         True             3175
False    False        False            3083
Name: count, dtype: int64


In [39]:
for c in ["ticket_no", "fare_conditions", "seat_no"]:
    df[c] = df[c].str.strip()

df["fare_conditions"] = df["fare_conditions"].str.title()
df["fare_conditions"]

0        Economy
1       Business
2        Economy
3        Economy
4        Economy
          ...   
6253     Economy
6254     Economy
6255     Economy
6256     Economy
6257     Economy
Name: fare_conditions, Length: 6258, dtype: object

In [40]:
df["seat_no"] = df["seat_no"].str.upper()
df["seat_no"]

0       45E
1       NaN
2       NaN
3       18C
4       NaN
       ... 
6253    NaN
6254    NaN
6255    32F
6256    NaN
6257    34F
Name: seat_no, Length: 6258, dtype: object

In [41]:
print("Full duplicate rows: ", df.duplicated().sum())
print("Duplicate (ticket_no, flight_id): ", df.duplicated(subset = ["ticket_no", "flight_id"]).sum())
df = df.drop_duplicates().drop_duplicates(subset = ["ticket_no", "flight_id"])

Full duplicate rows:  0
Duplicate (ticket_no, flight_id):  0


In [42]:
print("Rows now: ", len(df))

tickets = pd.read_csv(r"C:\Users\91766\Downloads\cleaned_data\6_tickets_clean.csv", dtype=str)
flights = pd.read_csv(r"C:\Users\91766\Downloads\cleaned_data\4_flights_clean.csv")

print(df["ticket_no"].head(3).tolist())
print(tickets["ticket_no"].head(3).tolist())
print(df["flight_id"].dtype, flights["flight_id"].dtype)

print("Bad tickets: ", (~df["ticket_no"].isin(tickets["ticket_no"])).sum())
print("Bad flights: ", (~df["flight_id"].isin(flights["flight_id"])).sum())

Rows now:  6258
['1000000000000', '1000000000001', '1000000000002']
['1000000000000', '1000000000001', '1000000000002']
int64 int64
Bad tickets:  0
Bad flights:  0


In [43]:
df["boarding_no"] = df["boarding_no"].astype("Int64")
print(df["boarding_no"].dropna().head(3).tolist())

[1, 1, 1]


In [44]:
df["boarding_time"] = pd.to_datetime(df["boarding_time"], errors = "coerce")
print(df["boarding_time"].isnull().sum())

3175


In [45]:
print(df["ticket_no"].str.match(r"^\d{13}$").all())
print(df["seat_no"].dropna().str.match(r"^\d{1,3}[A-Z]$").all())
print(df["fare_conditions"].value_counts())
print(df["boarding_no"].min(), df["boarding_no"].max())

True
True
fare_conditions
Economy     4653
Comfort      943
Business     662
Name: count, dtype: int64
1 15


In [46]:
print("Zero or negative prices: ", (df["price"] <= 0).sum())

Zero or negative prices:  0


In [47]:
print(df.groupby("fare_conditions")["price"].agg(["min", "mean", "max"]).round(0))

                     min     mean      max
fare_conditions                           
Business         15030.0  37357.0  59970.0
Comfort           7002.0  11597.0  16000.0
Economy           2500.0   5739.0   8999.0


In [48]:
for fc, g in df.groupby("fare_conditions"):
    q1, q3 = g["price"].quantile([0.25, 0.75]); iqr = q3 - q1
    n = ((g["price"] > q3 + 1.5*iqr) | (g["price"] < q1 - 1.5*iqr)).sum()
    print(fc, "outliers: ", n)

Business outliers:  0
Comfort outliers:  0
Economy outliers:  0


In [63]:
tickets = pd.read_csv(r"C:\Users\91766\Downloads\cleaned_data\6_tickets_clean.csv", dtype=str)
flights = pd.read_csv(r"C:\Users\91766\Downloads\cleaned_data\4_flights_clean.csv",
                      parse_dates=["scheduled_departure"])

bad_t = ~df["ticket_no"].isin(tickets["ticket_no"])
bad_f = ~df["flight_id"].isin(flights["flight_id"])
print(bad_t.sum(), bad_f.sum())

assert (bad_t | bad_f).sum() < len(df), "STOP: every row mismatched"
df = df[~(bad_t | bad_f)]

print("Tickets with no segment: ", (~tickets["ticket_no"].isin(df["ticket_no"])).sum())
print("Flights with no segment: ", (~flights["flight_id"].isin(df["flight_id"])).sum())

0 0


AssertionError: STOP: every row mismatched

In [50]:
# the 3 check in columns must be blank or filled together
print((df[["seat_no", "boarding_no", "boarding_time"]].isnull().nunique(axis =1) == 1).all())

True


In [51]:
# boarding must happen before the scheduled departure
m = df.merge(flights[["flight_id", "scheduled_departure", "status", "airplane_code"]], on = "flight_id")
x = m.dropna(subset = ["boarding_time"])
mins = (x["scheduled_departure"] - x["boarding_time"]).dt.total_seconds() / 60
print("Boarded after departure: ", (mins < 0).sum(), "| min/max minutes before : ", mins.min(), mins.max())

Boarded after departure:  0 | min/max minutes before :  nan nan


In [52]:
# which flight statuses have check ins?
print(m.groupby("status")["seat_no"].apply(lambda s: s.notna().sum()))

Series([], Name: seat_no, dtype: object)


In [53]:
# assigned seat's class vs the fare class paid
seats = pd.read_csv(r"C:\Users\91766\Downloads\cleaned_data\3_seats_clean.csv", dtype = str)
a = m.dropna(subset = ["seat_no"]).merge(seats, on = ["airplane_code", "seat_no"], how = "left", suffixes = ("", "_seat"))
print("Seat not in seat map: ", a["fare_conditions_seat"].isna().sum())
print("Fare class != seat class: ", (a["fare_conditions"] != a["fare_conditions_seat"]).sum(), "of", len(a))


Seat not in seat map:  0
Fare class != seat class:  0 of 0


In [54]:
# same seat given to two passengers on one flight
print("Seats double-assigned: ", a.duplicated(subset = ["flight_id", "seat_no"]).sum())

Seats double-assigned:  0


In [55]:
print(df.shape, df.isnull().sum().to_dict())

(0, 7) {'ticket_no': 0, 'flight_id': 0, 'fare_conditions': 0, 'price': 0, 'seat_no': 0, 'boarding_no': 0, 'boarding_time': 0}


In [56]:
print("Total revenue: ", round(df["price"].sum(),2))

Total revenue:  0.0


In [67]:
import os
out_dir = r"C:\Users\91766\Downloads\cleaned_data"
os.makedirs(out_dir, exist_ok=True)

df.to_csv(os.path.join(out_dir, "7_segments_clean.csv"),
          index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.2f")
print("Saved:", len(df), "rows")

Saved: 6258 rows


In [66]:
import os
print("segments rows now:", len(df))

for f in ["4_flights_clean.csv", "6_tickets_clean.csv"]:
    p = rf"C:\Users\91766\Downloads\cleaned_data\{f}"
    print(f, "| size:", os.path.getsize(p), "bytes | rows:", len(pd.read_csv(p)))

segments rows now: 6258
4_flights_clean.csv | size: 83478 bytes | rows: 900
6_tickets_clean.csv | size: 263119 bytes | rows: 4805


In [65]:
import os
import pandas as pd

base  = r"C:\Users\91766\Downloads"
raw   = os.path.join(base, "airline_booking_dataset", "7_segments.csv")
clean = os.path.join(base, "cleaned_data")

# reload everything from the file (does not depend on old cells)
df = pd.read_csv(raw, dtype={"ticket_no": str, "seat_no": str})
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
for c in ["ticket_no", "fare_conditions", "seat_no"]:
    df[c] = df[c].str.strip()
df["fare_conditions"] = df["fare_conditions"].str.title()
df["seat_no"] = df["seat_no"].str.upper()
df = df.drop_duplicates().drop_duplicates(subset=["ticket_no", "flight_id"])
df["boarding_no"]   = df["boarding_no"].astype("Int64")
df["boarding_time"] = pd.to_datetime(df["boarding_time"], errors="coerce")
print("rows loaded:", len(df))                                   # 6258

tickets = pd.read_csv(os.path.join(clean, "6_tickets_clean.csv"), dtype=str)
flights = pd.read_csv(os.path.join(clean, "4_flights_clean.csv"))
print("tickets rows:", len(tickets), "| flights rows:", len(flights))   # 4805, 900

bad_t = ~df["ticket_no"].isin(tickets["ticket_no"])
bad_f = ~df["flight_id"].isin(flights["flight_id"])
print("bad tickets:", bad_t.sum(), "| bad flights:", bad_f.sum())       # 0, 0
df = df[~(bad_t | bad_f)]

print("final rows:", len(df), "| revenue:", round(df["price"].sum(), 2))  # 6258, 62371057.97
out = os.path.join(clean, "7_segments_clean.csv")
df.to_csv(out, index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.2f")
print("Saved:", len(df), "rows")

rows loaded: 6258
tickets rows: 4805 | flights rows: 900
bad tickets: 0 | bad flights: 0
final rows: 6258 | revenue: 62371057.97
Saved: 6258 rows
